In [1]:
import torch, torchvision
import torch.nn as nn
import torch.optim as optim
from openai import OpenAI
import pandas as pd
import numpy as np
import os # get the API key with that
import json
from collections import defaultdict
import tiktoken # for counting tokens

# models
EMBEDDING_MODEL = "text-embedding-ada-002"
GPT_MODEL = "gpt-4"

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [10]:
response = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': 'You have sentiment analysis task where there are two labels: Positive and Negative. You have to give probability of sentiment of input.'},
        {'role': 'user', 'content': 'p("Positive" | "Input: nothing Sentiment:") = ?, p("Negative" | "Input: nothing Sentiment:") = ?'}
    ],
    model=GPT_MODEL
)

print(response.choices[0].message.content)

Without specific input text, it is not possible to compute the probability of sentiment. In sentiment analysis, the text input plays a vital role in predicting its sentiment. But generally, if you have no information at all, you might assume the probabilities are equal. So, in that case it would be: 

p("Positive" | "Input: nothing Sentiment:") = 0.5, 
p("Negative" | "Input: nothing Sentiment:") = 0.5

However, it's likely more accurate to say the sentiment is undefined with no input:

p("Positive" | "Input: nothing Sentiment:") = undefined, 
p("Negative" | "Input: nothing Sentiment:") = undefined. 

In a real model, the task would be to learn these probability distributions from a training dataset of examples where the sentiment is known.


In [14]:
prompt = """
Input: I hate this movie. Sentiment: Negative
Input: I love this movie. Sentiment: Positive
Input: N/A Sentiment: Positive
Input: N/A Sentiment: Negative
Input: nothing Sentiment: Positive
Input: nothing Sentiment: Negative
Input: I like eggs. Sentiment:
p("Positive" | "Input: I like eggs Sentiment:") = ?, p("Negative" | "Input: I like eggs Sentiment:") = ?
"""

prompt

'\nInput: I hate this movie. Sentiment: Negative\nInput: I love this movie. Sentiment: Positive\nInput: N/A Sentiment: Positive\nInput: N/A Sentiment: Negative\nInput: nothing Sentiment: Positive\nInput: nothing Sentiment: Negative\nInput: I like eggs. Sentiment:\np("Positive" | "Input: I like eggs Sentiment:") = ?, p("Negative" | "Input: I like eggs Sentiment:") = ?\n'

In [15]:
response = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': 'You have sentiment analysis task where there are two labels: Positive and Negative. You have to give probability of sentiment of input.'},
        {'role': 'user', 'content': prompt}
    ],
    model=GPT_MODEL
)

print(response.choices[0].message.content)

Considering that 'I like eggs' is a neutral statement but has a positive connotation due to the expression of liking something:

p("Positive" | "Input: I like eggs Sentiment:") = 0.6 
p("Negative" | "Input: I like eggs Sentiment:") = 0.4
